Use Colab as the Runtime Kernel

Train in Google Colab GPU
I select Python 3 (ipykernel)

In [1]:
import os
import torch

In [2]:
torch.cuda.is_available()

True

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import pandas as pd
path = "/content/drive/MyDrive/Colab Notebooks/Files/classifier_train.csv"
pd.read_csv(path).head(2)

,text,label
0,**Q3 Financial Report & M&A Target Assessment*...,Confidential
1,**FOR IMMEDIATE RELEASE** \n**Contact:** \nJ...,Public


In [ ]:
%pip install transformers datasets accelerate tiktoken

In [10]:
import os
import torch
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    AutoConfig,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

import pandas as pd
from datasets import Dataset

BASE_MODEL = "MoritzLaurer/ModernBERT-base-zeroshot-v2.0"
HF_REPO_ID = "KenzyXriah/modernbert-doc-classifier"
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/Files/checkpoints"
HF_TOKEN = "enter your hf token"


def train_model(dataset: Dataset, id2label: dict, epochs: int = 3, batch_size: int = 8, max_length: int = 1024, output_dir: str = None)-> None:
    """
    Fine-tunes the base ModernBERT model on a provided dataset and saves the resulting weights to the configured directory.
    
    Args:
        dataset (Dataset): Hugging Face Dataset with 'text' and 'label' (int) columns.
        id2label (dict): Mapping from integer label IDs to string class names.
                         e.g., {0: "Public", 1: "Internal", 2: "Confidential", 3: "Restricted"}
        epochs (int): Number of training epochs.
        batch_size (int): Training batch size.
    """
    target_output_dir = output_dir if output_dir else OUTPUT_DIR
    
    num_labels = len(id2label)
    label2id = {v: k for k, v in id2label.items()}
    
    # try:
    #     AutoConfig.from_pretrained(HF_REPO_ID, token=HF_TOKEN)
    #     model_source = HF_REPO_ID
    #     print(f"Found existing model on Hub! Resuming continuous training from {model_source}...")
    # except Exception:
    #     model_source = BASE_MODEL
    #     print(f"No existing model on Hub. Starting fresh training from {model_source}...")

    # new training
    model_source = BASE_MODEL
    print(f"Loading model and tokenizer from {model_source}...")
    tokenizer = AutoTokenizer.from_pretrained(model_source, token=HF_TOKEN)
    
    model_config = AutoConfig.from_pretrained(model_source, token=HF_TOKEN)
    model_config.num_labels = num_labels
    model_config.id2label = id2label
    model_config.label2id = label2id
    
    model = AutoModelForSequenceClassification.from_pretrained(
        model_source, 
        config=model_config,
        token=HF_TOKEN,
        ignore_mismatched_sizes=True
    ).to(torch.float32)
    
    print("Tokenizing dataset...")
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=max_length)
        
    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    training_args = TrainingArguments(
        output_dir=target_output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        save_strategy="epoch",
        logging_steps=10,
        push_to_hub=False, #True,
        hub_model_id=HF_REPO_ID,
        hub_token=HF_TOKEN
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        processing_class=tokenizer,
        data_collator=data_collator
    )
    
    resume = True if os.path.exists(target_output_dir) else False
    
    try:
        trainer.train(resume_from_checkpoint=resume)
    except ValueError:
        trainer.train()
    
    # print(f"Pushing fine-tuned model to Hugging Face Hub ({HF_REPO_ID})...")
    # trainer.push_to_hub()
    trainer.save_model(target_output_dir)
    
    print(f"Training complete! Model successfully pushed to {HF_REPO_ID}.")



def main():
    label_names = ["Public", "Internal", "Confidential", "Restricted"]
    label2id = {name: idx for idx, name in enumerate(label_names)}
    id2label = {idx: name for idx, name in enumerate(label_names)}

    print("Loading training data...")
    df = pd.read_csv(path).dropna(subset=["text", "label"])
        
    df["label"] = df["label"].map(label2id)

    dataset = Dataset.from_pandas(df[["text", "label"]])
    
    print("Starting full training pipeline...")
    train_model(dataset, id2label)

main()


Loading training data...
Starting full training pipeline...
Loading model and tokenizer from MoritzLaurer/ModernBERT-base-zeroshot-v2.0...


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: MoritzLaurer/ModernBERT-base-zeroshot-v2.0
Key               | Status   |                                                                                     
------------------+----------+-------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2, 768]) vs model:torch.Size([4, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2]) vs model:torch.Size([4])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Tokenizing dataset...


Map:   0%|          | 0/1398 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete! Model successfully pushed to KenzyXriah/modernbert-doc-classifier.


In [12]:
pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Files/classifier_test.csv")

,text,label
0,Check out our latest blog post on how digital ...,Public
1,Our mobile app is now available on both iOS an...,Public
2,Reminder: The company town hall meeting is sch...,Internal
3,HR Update: The new employee wellness program l...,Internal
4,The new office seating arrangement takes effec...,Internal
...,...,...
345,# HR Manual: Company-Wide Policy Overview\n\n#...,Internal
346,# HR Manual: Company-Wide Policy\n\n## Introdu...,Internal
347,The Assembly was called by the Long Parliamen...,Public
348,**Confidential Corporate Document** \n**Q3 Fi...,Confidential
